# APIM ❤️ Microsoft Web IQ

## Track Microsoft Web IQ REST and MCP usage with Azure API Management

Route [Microsoft Web IQ](https://webiq.microsoft.ai/documentation/overview/) REST and MCP requests through Azure API Management (APIM), block Browse at the gateway, and attribute request volume and latency to APIM subscriptions. Callers provide a Web IQ API key or Entra ID token; APIM does not create or inject an upstream credential. This notebook deploys the shared gateway and exercises both interfaces.

**Audience:** Developers and platform teams operating web-grounded applications.

**Prerequisites:**

- Microsoft Web IQ limited-access approval and an API key from Web IQ Profile Management.
- Python 3.12+, the repository environment installed with `uv sync`, and VS Code with the Jupyter extension.
- Azure CLI installed and authenticated.
- An Azure subscription with Contributor + RBAC Administrator, or Owner, permissions.

**Learning goals:**

- Pass a caller-provided Web IQ API key through APIM without storing it at the gateway.
- Use APIM subscriptions as consumer identities.
- Block the Browse REST operation and MCP tool before either reaches Web IQ.
- Discover and invoke Web IQ tools over streamable HTTP MCP.
- Optionally authenticate Web IQ with a Microsoft Entra ID app-only token.
- Analyze upstream response text with the standard Hate, Sexual, Violence, and SelfHarm categories using APIM’s system managed identity.
- Inspect Content Safety response headers and query usage and moderation metrics in Application Insights.

See the [architecture diagrams](README.md#architecture) in the lab README.

> APIM does not store the Web IQ credential. This notebook keeps the API key only in kernel memory and sends it in the `x-apikey` request header.


## Outline

1. Configure the lab and verify Azure CLI access.
2. Deploy APIM, Content Safety, Application Insights, and Log Analytics.
3. Send Web IQ REST requests through APIM subscriptions.
4. Optionally call Web IQ with Microsoft Entra ID.
5. Discover and invoke Web IQ tools over MCP.
6. Verify that APIM blocks Browse over MCP.
7. Query request, response, latency, and MCP tool metrics, then exercise the REST Browse denial.


<a id='initialize'></a>
### 0️⃣ Initialize notebook variables

The caller-provided Web IQ key is read from `WEBIQ_API_KEY` when available; otherwise the notebook prompts without echoing it. APIM clients receive separate subscription keys for usage attribution.


In [1]:
from __future__ import annotations

import json
import os
import sys
import time
from getpass import getpass
from dotenv import load_dotenv

sys.path.insert(1, '../../shared')
import utils

load_dotenv()
deployment_name = 'web-iq'
resource_group_name = f'lab-{deployment_name}'
resource_group_location = 'westus2'

apim_sku = 'Basicv2'
web_iq_api_path = 'web-iq'
apim_subscriptions_config = [
    {'name': 'research-team', 'displayName': 'Research Team'},
    {'name': 'support-team', 'displayName': 'Support Team'},
]

web_iq_api_key = os.getenv('WEBIQ_API_KEY') or getpass('Microsoft Web IQ API key: ')
if not web_iq_api_key:
    raise ValueError('Set WEBIQ_API_KEY or enter a Web IQ API key when prompted.')

utils.print_ok('Notebook initialized')


✅ Notebook initialized ⌚ 09:46:48.263054 


<a id='azure-cli'></a>
### 1️⃣ Verify Azure CLI and the active subscription

Confirm that subsequent deployment commands target the intended tenant and subscription.


In [2]:
output = utils.run('az account show', 'Retrieved Azure account', 'Failed to get the current Azure account')

if not output.success or not output.json_data:
    raise RuntimeError('Authenticate with Azure CLI by running az login, then rerun this cell.')

current_user = output.json_data['user']['name']
tenant_id = output.json_data['tenantId']
subscription_id = output.json_data['id']
utils.print_info(f'Current user: {current_user}')
utils.print_info(f'Tenant ID: {tenant_id}')
utils.print_info(f'Subscription ID: {subscription_id}')


⚙️ Running: az account show 
✅ Retrieved Azure account ⌚ 09:46:53.276967 :3s]
👉🏽 Current user: jacwang@microsoft.com
👉🏽 Tenant ID: 16b3c013-d300-468d-ac64-7eda0820b6d3
👉🏽 Subscription ID: 6025ba02-1dfd-407f-b358-88f811c7c7aa


<a id='deploy'></a>
### 2️⃣ Deploy the lab with Bicep

The deployment creates APIM, two APIM subscriptions, Content Safety, Application Insights, and Log Analytics. APIM’s system-assigned managed identity receives the Cognitive Services User role scoped to Content Safety. Key authentication on Content Safety is disabled. It does not deploy or store a Web IQ credential. The API policy permits authenticated Web Search and MCP traffic, but returns `403 Forbidden` for Browse through either interface.

Content Safety is shown as optional in the architecture diagrams and is always enabled by this deployment. Only the four standard text moderation categories are used. Redeploy this cell to update an existing lab, and allow several minutes for a new role assignment to propagate.


In [3]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'apimSku': {'value': apim_sku},
        'apimSubscriptionsConfig': {'value': apim_subscriptions_config},
        'webIqApiPath': {'value': web_iq_api_path},
    },
}

with open('params.json', 'w', encoding='utf-8') as parameters_file:
    json.dump(bicep_parameters, parameters_file)

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} '
    '--template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed",
)
if not output.success:
    raise RuntimeError('Deployment failed. Review the Azure CLI output above.')


⚙️ Running: az group show --name lab-web-iq 
👉🏽 Using existing resource group 'lab-web-iq'
⚙️ Running: az deployment group create --name web-iq --resource-group lab-web-iq --template-file main.bicep --parameters params.json 
✅ Deployment 'web-iq' succeeded ⌚ 09:48:51.761284 :48s]


<a id='outputs'></a>
### 3️⃣ Retrieve gateway details

Only the APIM subscription keys are displayed, masked to their final four characters. The caller-provided Web IQ key remains only in notebook kernel memory.


In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} --resource-group {resource_group_name}',
    f"Retrieved deployment '{deployment_name}'",
    f"Failed to retrieve deployment '{deployment_name}'",
)
if not output.success or not output.json_data:
    raise RuntimeError('Could not retrieve the deployment outputs.')

apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM gateway URL')
application_insights_name = utils.get_deployment_output(output, 'applicationInsightsName', 'Application Insights name')
content_safety_endpoint = utils.get_deployment_output(output, 'contentSafetyEndpoint', 'Content Safety endpoint')
web_iq_api_path = utils.get_deployment_output(output, 'webIqApiPath', 'Web IQ API path')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))

for subscription in apim_subscriptions:
    utils.print_info(f"{subscription['displayName']}: ****{subscription['key'][-4:]}")


<a id='search'></a>
### 4️⃣ Send Web Search requests through APIM

Each team uses its own APIM subscription key for usage attribution and sends its Web IQ API key in `x-apikey`. APIM strips the subscription credential, forwards the caller-provided Web IQ credential, and emits usage metrics. The compact display follows the official [Web Response schema](https://webiq.microsoft.ai/documentation/api-reference/web/#web-response), including query signals, instrumentation availability, and per-result metadata.

Each REST JSON response includes a `contentSafety` object. The final table shows **Hate**, **Sexual**, **Violence**, and **SelfHarm** scores beside the scan status, chunk count, and moderation latency. The same telemetry remains available in `x-content-safety-*` response headers. Scores use 0/2/4/6 and do not automatically block flagged text. A failed or oversized scan returns `502` without the upstream content. See [standard response text moderation](README.md#standard-response-text-moderation) for limits and MCP behavior.


In [ ]:
import requests
import pandas as pd
from content_safety import content_safety_from_response as content_safety_telemetry, content_safety_row


def summarize_web_response(data: dict, content_preview_length: int = 280) -> dict:
    web_results = []
    for result in data.get('webResults', []):
        content = (result.get('content') or '').replace('\n', ' ')
        web_results.append({
            'title': result.get('title'),
            'url': result.get('url'),
            'contentPreview': content[:content_preview_length],
            'crawledAt': result.get('crawledAt'),
            'lastUpdatedAt': result.get('lastUpdatedAt'),
            'language': result.get('language'),
            'isAdult': result.get('isAdult'),
            'contentTier': result.get('contentTier'),
            'clickUrl': result.get('clickUrl'),
            'instrumentationSuffix': result.get('instrumentationSuffix'),
        })
    return {
        'traceId': data.get('traceId'),
        'querySignals': data.get('querySignals'),
        'hasInstrumentationClickBase': bool(data.get('instrumentationClickBase')),
        'hasInstrumentationCitationBase': bool(data.get('instrumentationCitationBase')),
        'webResults': web_results,
    }

search_url = f'{apim_resource_gateway_url}/{web_iq_api_path}/search/web'
workloads = [
    (
        "research-team",
        "How does Azure API Management support AI gateways?",
    ),
    (
        "support-team",
        "What is Microsoft Web IQ?",
    ),
    (
        "research-team",
        "Latest news in the US",
    ),
    (
        "research-team",
        "News from the last week",
    ),
    (
        "research-team",
        "Recap this week's tech news",
    ),
    (
        "research-team",
        (
            "How is the US Federal Reserve's interest rate policy affecting the "
            "housing market?"
        ),
    ),
    (
        "research-team",
        (
            "Classify the company services for License Corporation - "
            "licensecorporation.com"
        ),
    ),
    (
        "research-team",
        "Mayo Clinic Kaiser Permanente AI agents deployment 2025",
    ),
    (
        "research-team",
        (
            "How is the U.S. Federal Reserve's interest rate policy affecting the "
            "housing market?"
        ),
    ),
    (
        "support-team",
        (
            "What recent regulatory enforcement actions involved banks using AI, "
            "algorithmic decisioning, model risk management, or customer "
            "communications?"
        ),
    ),
    (
        "support-team",
        "License Corporation licensecorporation.com company services",
    ),
]
subscription_by_name = {item['name']: item for item in apim_subscriptions}
request_summary = []

for subscription_name, query in workloads:
    subscription = subscription_by_name[subscription_name]
    started = time.perf_counter()
    response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'x-apikey': web_iq_api_key,
            'content-type': 'application/json',
        },
        json={
            'query': query,
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=180,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000
    utils.print_response_code(response)

    try:
        data = response.json()
    except requests.JSONDecodeError:
        data = {'rawResponse': response.text[:500]}

    web_response_summary = summarize_web_response(data) if isinstance(data, dict) else {'webResults': []}
    web_response_summary['contentSafety'] = content_safety_telemetry(response)
    results = web_response_summary['webResults']
    request_summary.append({
        'subscription': subscription_name,
        'query': query,
        'search_url': search_url,
        'status': response.status_code,
        'latency_ms': round(elapsed_ms, 1),
        'content_safety': web_response_summary['contentSafety'],
        'results': len(results),
        'trace_id': web_response_summary.get('traceId'),
        'freshness': (web_response_summary.get('querySignals') or {}).get('freshness'),
        #'click_instrumentation': web_response_summary.get('hasInstrumentationClickBase', False),
        #'citation_instrumentation': web_response_summary.get('hasInstrumentationCitationBase', False),
    })

    print(json.dumps(web_response_summary, indent=2))
    if not response.ok:
        print(json.dumps(data, indent=2)[:2000])

content_safety_results = pd.DataFrame([
    {
        'subscription': item['subscription'],
        'query': item['query'],
        **content_safety_row(item['content_safety']),
    }
    for item in request_summary
])
content_safety_results


<a id='entra-id'></a>
### 5️⃣ Optional: authenticate Web IQ with Microsoft Entra ID

Web IQ recommends [Entra ID app-only authentication](https://webiq.microsoft.ai/documentation/authentication/#entra-id) for production workloads. Create an app registration and client credential, then bind its **Application (client) ID** in Web IQ Profile Management. Set `WEBIQ_TENANT_ID`, `WEBIQ_CLIENT_ID`, and `WEBIQ_CLIENT_SECRET` in your environment before running this cell. The token scope is `https://api.microsoft.ai/.default`.

The APIM subscription key still identifies the consuming team. When APIM sees the bearer token, it removes any `x-apikey` and lets Web IQ validate the Entra token. If the environment variables are absent, this optional step is skipped.


In [ ]:
from msal import ConfidentialClientApplication

entra_mcp_headers = None
entra_settings = {
    'tenant_id': os.getenv('WEBIQ_TENANT_ID'),
    'client_id': os.getenv('WEBIQ_CLIENT_ID'),
    'client_secret': os.getenv('WEBIQ_CLIENT_SECRET'),
}

if not all(entra_settings.values()):
    utils.print_info(
        'Optional Entra ID call skipped. Set WEBIQ_TENANT_ID, WEBIQ_CLIENT_ID, and WEBIQ_CLIENT_SECRET to run it.'
    )
else:
    entra_client = ConfidentialClientApplication(
        client_id=entra_settings['client_id'],
        client_credential=entra_settings['client_secret'],
        authority=f"https://login.microsoftonline.com/{entra_settings['tenant_id']}",
    )
    token_result = entra_client.acquire_token_for_client(
        scopes=['https://api.microsoft.ai/.default']
    )
    if 'access_token' not in token_result:
        raise RuntimeError(token_result.get('error_description', 'Could not acquire a Web IQ access token.'))

    entra_mcp_headers = {
        'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
        'Authorization': f"Bearer {token_result['access_token']}",
    }
    entra_response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
            'Authorization': f"Bearer {token_result['access_token']}",
            'content-type': 'application/json',
        },
        json={
            'query': 'What is Microsoft Web IQ?',
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=180,
    )
    utils.print_response_code(entra_response)
    entra_data = entra_response.json()
    entra_summary = summarize_web_response(entra_data)
    entra_summary['contentSafety'] = content_safety_telemetry(entra_response)
    print(json.dumps(entra_summary, indent=2))

if all(entra_settings.values()):
    display(pd.DataFrame([content_safety_row(entra_summary['contentSafety'])]))


<a id='metrics'></a>
### 9️⃣ Query usage metrics in Application Insights

Custom metrics can take several minutes to arrive. If the tables are empty, wait briefly and rerun these cells. The first query reports request volume and blocked calls by APIM subscription, operation, authentication mode, and MCP tool.

The helper uses the installed `az` command and its signed-in account. It sends KQL in a temporary JSON file through `az rest`, preserving quotes and line breaks on Windows, Linux, and macOS. The notebook kernel does not need the `azure.cli` Python package. Query failures raise an error; a successful query with no matching telemetry returns an empty table.


In [ ]:
import json
import shutil
import subprocess
import tempfile
from pathlib import Path

import pandas as pd


def query_application_insights(kql: str) -> pd.DataFrame:
    az = shutil.which('az')
    if not az:
        raise RuntimeError('Azure CLI was not found on PATH. Install it and restart the notebook kernel.')

    def run_az(*arguments):
        result = subprocess.run(
            [az, *arguments, '--output', 'json', '--only-show-errors'],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(
                f'Application Insights query failed:\n{result.stderr.strip() or result.stdout.strip()}'
            )
        return json.loads(result.stdout.lstrip('\ufeff'))

    app_id = run_az(
        'resource', 'show', '--name', application_insights_name,
        '--resource-group', resource_group_name,
        '--resource-type', 'Microsoft.Insights/components',
        '--api-version', '2020-02-02', '--query', 'properties.AppId',
    )
    if not app_id:
        raise RuntimeError('The Application Insights resource did not return an AppId.')

    # File input preserves KQL verbatim, including when az is a Windows .cmd launcher.
    with tempfile.TemporaryDirectory(prefix='web-iq-query-') as directory:
        body_file = Path(directory) / 'query.json'
        body_file.write_text(json.dumps({'query': kql, 'timespan': 'PT1H'}), encoding='utf-8')
        data = run_az(
            'rest', '--method', 'post',
            '--url', f'https://api.applicationinsights.io/v1/apps/{app_id}/query',
            '--resource', 'https://api.applicationinsights.io',
            '--headers', 'Content-Type=application/json',
            '--body', f'@{body_file}',
        )

    tables = data.get('tables')
    if not tables:
        raise RuntimeError('Application Insights query returned no result tables.')
    table = tables[0]
    utils.print_ok('Application Insights query succeeded')
    return pd.DataFrame(
        table.get('rows', []),
        columns=[column['name'] for column in table.get('columns', [])],
    )

usage_query = r'''
customMetrics
| where timestamp > ago(1h) and name in ('Web IQ Requests', 'Web IQ Blocked Requests')
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend McpTool = tostring(dimensions['MCP Tool'])
| summarize Requests = sumif(value, name == 'Web IQ Requests'), BlockedRequests = sumif(value, name == 'Web IQ Blocked Requests')
    by SubscriptionId, OperationId, Authentication, McpTool
| order by Requests desc
'''

usage_df = query_application_insights(usage_query)
usage_df


The latency query groups completed calls by consumer, operation, MCP tool, and upstream HTTP status. This makes throttling and service errors visible without capturing request or response bodies.


In [ ]:
latency_query = r'''
customMetrics
| where timestamp > ago(1h) and name == 'Web IQ Latency'
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend McpTool = tostring(dimensions['MCP Tool'])
| extend StatusCode = tostring(dimensions['Status Code'])
| summarize Calls = count(), AverageLatencyMs = round(avg(value), 1), P95LatencyMs = round(percentile(value, 95), 1)
    by SubscriptionId, OperationId, Authentication, McpTool, StatusCode
| order by Calls desc
'''

latency_df = query_application_insights(latency_query)
latency_df


### Query Content Safety outcomes

This query counts complete analyses, responses with no text, and failures by consumer and operation. Moderation latency is reported separately from total gateway latency. Response category scores are available on direct gateway HTTP responses; analyzed text is not added to these metrics.


In [ ]:
content_safety_query = r"""
customMetrics
| where timestamp > ago(1h)
| where name == 'Web IQ Content Safety Responses'
| extend SubscriptionId = tostring(customDimensions['Subscription ID']),
         OperationId = tostring(customDimensions['Operation ID']),
         SafetyStatus = tostring(customDimensions['Content Safety Status'])
| summarize Responses = sum(value) by SubscriptionId, OperationId, SafetyStatus
| order by Responses desc
"""

content_safety_df = query_application_insights(content_safety_query)
content_safety_df


### Exercise — verify that Browse is blocked

Use the second APIM subscription to call `/browse`. Before running the next cell, predict the status and response body. The request should return APIM's structured `403` without reaching Web IQ. After telemetry arrives, rerun the usage query above to see `browse-url` with one blocked request.


In [ ]:
# Answer scaffold: change target_url to verify that every Browse target is blocked.
def browse_with_subscription(target_url: str, subscription_index: int = 1) -> requests.Response:
    subscription = apim_subscriptions[subscription_index]
    response = requests.post(
        f'{apim_resource_gateway_url}/{web_iq_api_path}/browse',
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'x-apikey': web_iq_api_key,
            'content-type': 'application/json',
        },
        json={
            'url': target_url,
            'contentFormat': 'markdown',
            'maxLength': 3000,
            'liveCrawl': 'fallback',
        },
        timeout=30,
    )
    utils.print_response_code(response)
    print(json.dumps(response.json(), indent=2))
    return response

browse_response = browse_with_subscription('https://news.microsoft.com/source/')
assert browse_response.status_code == 403
assert browse_response.json()['errorCode'] == 'BrowseOperationBlocked'


In [24]:
# Answer scaffold: change target_url to verify that every Browse target is blocked.
def search_with_subscription(subscription_index: int = 1) -> requests.Response:
    subscription = apim_subscriptions[subscription_index]
    response = requests.post(
        f'{apim_resource_gateway_url}/{web_iq_api_path}/search/web',
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'x-apikey': web_iq_api_key,
            'content-type': 'application/json',
        },
       json={
            'query': "reddit r/frenworld",
            'maxResults': 3,
            'maxLength': 1000,
            'contentFormat': 'markdown',
        },
        timeout=30,
    )
    utils.print_response_code(response)
    print(json.dumps(response.json(), indent=2))
    return response

response = search_with_subscription()

Response status: 200 - OK
{
  "webResults": [
    {
      "title": "index - fnafworld",
      "url": "https://www.reddit.com/r/fnafworld/wiki/index/",
      "content": "# index - fnafworld\n\n\nr/fnafworld\n# Adveeeeeeeenture!\n\n\n## r/ fnafworld\n\n\nJoin\n\"index\" does not exist\nAbout Community\n/r/FNAFWorld is a sub for Scott Cawthon's game, FNaF World, this game was released on the day 21 of January in 2016, but remastered by Scott Cawthon himself and not re-released yet.\n\n\nCreated Sep 15, 2015\nRestricted\n496\nMembers\n3\nOnline\nFilter by flair\nVideo\nText\nGameplay\nWall-O-Text\nModerators\nModerator list hidden. Learn More\n\n\nUser Agreement Privacy policy\nContent policy Moderator Code of Conduct\nReddit, Inc. \u00a9 2023. All rights reserved.\n\n\nBack to Top\n<!-- r/fnafworld: /r/FNAFWorld is a sub for Scott Cawthon's game, FNaF World, this game was released on the day 21 of January in 2016, but remastered by \u2026 -->",
      "lastUpdatedAt": "2015-09-15T00:00:00"

### Pitfalls and extensions

- **Telemetry delay:** Application Insights custom metrics are not immediate; rerun the queries after a few minutes.
- **Credential boundaries:** Clients send an APIM subscription key plus either a Web IQ `x-apikey` or bearer token. APIM strips its subscription credential before forwarding and never stores the Web IQ API key.
- **Metric cardinality:** Keep dimensions bounded. Search text, URLs, trace IDs, and client IP addresses are intentionally excluded.
- **Policy scope:** The block checks APIM's stable REST operation ID (`browse-url`) and the MCP `tools/call` name (`browse`). Remove or revise the `choose` block in [policy.xml](policy.xml) only when Browse is approved for your consumers.
- **MCP reuse:** The `list_web_iq_tools` and `call_web_iq_tool` helpers can be reused by agent code that supplies the same APIM and Web IQ credentials.
- **Two kinds of instrumentation:** These APIM metrics measure gateway usage. Web IQ [instrumentation](https://webiq.microsoft.ai/documentation/instrumentation/) separately records which citations an LLM uses and which links users click.
- **Extension:** Add APIM rate limits per subscription, or proxy other Web IQ verticals after confirming that your Web IQ account permits them.

- **Moderation and MCP:** Responses are buffered for complete text analysis. Finite MCP POST/SSE results keep their framing; the optional long-lived MCP GET stream returns `405`. Empty acknowledgments report `no-text`.
- **Moderation limits:** Extracted text is limited to 100,000 UTF-16 code units per response and split into overlapping chunks. `502` with `ContentSafetyAnalysisFailed` means the gateway could not complete analysis; check the safety status header, resource quota, and managed-identity role before retrying.


<a id='portal'></a>
### View metrics in the Azure portal

Open the deployed Application Insights resource, select **Metrics**, choose the `web-iq` custom namespace, and select **Web IQ Requests**, **Web IQ Blocked Requests**, or **Web IQ Latency**. Split by **Subscription ID**, **Operation ID**, **Authentication**, **MCP Tool**, or **Status Code**.

Select **Web IQ Content Safety Responses** or **Web IQ Content Safety Latency** and split by **Content Safety Status** to inspect moderation outcomes.


<a id='clean-up'></a>
### 🗑️ Clean up resources

Run [clean-up-resources.ipynb](clean-up-resources.ipynb) when finished to remove the resource group and avoid further charges.
